In [5]:
# 1. Replace print statements with logging (INFO, WARNING, ERROR).
import logging
logging.info("CSV file loaded successfully")#normal
logging.warning("Some rows were skipped")#problem
logging.error("Could not open the CSV file")#error
print("logging statements")#print


INFO:root:CSV file loaded successfully
ERROR:root:Could not open the CSV file


logging statements


In [ ]:
# 2. Write a log file per run with a timestamp.
import logging
from datetime import datetime
log_file = datetime.now().strftime("run_%Y%m%d_%H%M%S.log")#year month date hour minute second
logging.basicConfig(filename=log_file, level=logging.INFO)#filename with the current timestamp, such as run_20261123_143025.log.

logging.info("CSV processing started")#write messages and above to that file.
print(log_file)

INFO:root:CSV processing started


run_20261123_115106.log


In [1]:
# 3. Move paths, thresholds and mappings into config.yaml and read it in the notebook.
%pip install pyyaml
import yaml 

with open("config.yaml") as file:
    config = yaml.safe_load(file)
orders_path = config["paths"]["orders"]
max_rejected = config["thresholds"]["max_rejected_fraction"]
column_mapping = config["mappings"]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 4. Write the clean output as CSV and JSON.
import csv
import json
with open("olist_mini/orders.csv", "r") as f:
    r=csv.DictReader(f)
    rows=list(r)
    f=r.fieldnames
with open("newCsv.csv","w+") as c:
    w=csv.DictWriter(c,fieldnames=f)
    w.writeheader()
    w.writerows(rows);c.seek(0)
    #rr=c.DictReader(c)
    for e,i in enumerate(csv.DictReader(c)):
        print(i)
        if(e>5):
            break
print("......................................................")
with open("newJson.json","w+") as j:
    json.dump(rows, j, indent=2) 
    j.seek(0)
    data = json.load(j)
    print(data[:5])



{'order_id': 'bfbd0f9bdef84302105ad712db648a6c', 'customer_id': '86dc2ffce2dfff336de2f386a786e574', 'order_status': 'delivered', 'order_purchase_timestamp': '2016-09-15 12:16:38', 'order_approved_at': '2016-09-15 12:16:38', 'order_delivered_carrier_date': '2016-11-07 17:11:53', 'order_delivered_customer_date': '2016-11-09 07:47:38', 'order_estimated_delivery_date': '2016-10-04 00:00:00'}
{'order_id': '8b388d845a3e8700444b971fa2bef902', 'customer_id': '00474d2582fd72663036795b7ab8cfc1', 'order_status': 'delivered', 'order_purchase_timestamp': '2016-10-04 22:33:44', 'order_approved_at': '2016-10-06 03:10:59', 'order_delivered_carrier_date': '2016-10-28 13:28:35', 'order_delivered_customer_date': '2016-11-08 14:03:58', 'order_estimated_delivery_date': '2016-12-06 00:00:00'}
{'order_id': 'd2945e686c883978378a10e49be7fb24', 'customer_id': '73cb441c896a460a30affe43bff3df82', 'order_status': 'delivered', 'order_purchase_timestamp': '2016-10-06 19:38:12', 'order_approved_at': '2016-10-07 10:11

In [26]:
# 5. Write a run summary JSON (counts, duration).
import csv
import json
import time
clean_rows=[];bad_records=[]
start_time = time.time()
with open("olist_mini/orders.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        if all(row.get(column) for column in ["order_id", "customer_id", "order_status"]):
            clean_rows.append(row)
        else:
            row["reason"] = "Missing required value"
            bad_records.append(row)
summary = {
    "total_rows": len(clean_rows) + len(bad_records),
    "clean_rows": len(clean_rows),
    "rejected_rows": len(bad_records),
    "duration_seconds": round(time.time() - start_time, 2)
}
with open("run_summary.json", "w") as file:
    json.dump(summary, file, indent=2)

In [27]:
# 6. Keep a manifest of processed files and skip them next time.
import json
from pathlib import Path

manifest_path = Path("manifest.json")
input_file = "olist_mini/orders.csv"

# Load existing manifest, or start with an empty list
if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)
else:
    manifest = {"processed_files": []}

# Skip files already recorded
if input_file in manifest["processed_files"]:
    print(f"Skipping already processed file: {input_file}")
else:
    # Put your file-processing code here
    print(f"Processing: {input_file}")

    manifest["processed_files"].append(input_file)
    with manifest_path.open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)

Processing: olist_mini/orders.csv


In [28]:
# 7. Run the whole pipeline from one notebook cell using config.yaml.
import csv
import json
import time
from pathlib import Path
import yaml


def run_pipeline(config):
    start = time.time()
    input_file = config["input_file"]
    manifest_path = Path(config["manifest"])

    # Load manifest
    if manifest_path.exists():
        with manifest_path.open(encoding="utf-8") as f:
            manifest = json.load(f)
    else:
        manifest = {"processed_files": []}

    if input_file in manifest["processed_files"]:
        print(f"Skipping already processed file: {input_file}")
        return

    # Read and validate rows
    clean_rows = []
    bad_records = []
    required = ["order_id", "customer_id", "order_status"]

    with open(input_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []

        for row in reader:
            if all(row.get(column) for column in required):
                clean_rows.append(row)
            else:
                row["reason"] = "Missing required value"
                bad_records.append(row)

    total_rows = len(clean_rows) + len(bad_records)
    rejected_fraction = len(bad_records) / total_rows if total_rows else 0

    if rejected_fraction > config["max_rejected_fraction"]:
        raise ValueError(f"Too many rejected rows: {rejected_fraction:.1%}")

    # Write clean CSV
    with open(config["clean_csv"], "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(clean_rows)

    # Write clean JSON
    with open(config["clean_json"], "w", encoding="utf-8") as f:
        json.dump(clean_rows, f, indent=2)

    # Update manifest after successful processing
    manifest["processed_files"].append(input_file)
    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(f"Done: {len(clean_rows)} clean, {len(bad_records)} rejected "
          f"in {time.time() - start:.2f}s")


with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

run_pipeline(config)

Skipping already processed file: olist_mini/orders.csv


In [29]:
# 8. Lint your .py modules with ruff (or flake8) and fix the warnings.
%pip install ruff
!ruff check .

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/10.4 MB 8.5 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/10.4 MB 13.1 MB/s eta 0:00:01
   ------------------------------- -------- 8.1/10.4 MB 15.3 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 14.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


invalid-syntax: Expected a statement
 --> day11.ipynb:cell 5:1:1
  |
1 | & "C:\Users\Harini\AppData\Local\Programs\Python\Python312\python.exe" -m pip install pandas
  | ^

invalid-syntax: Got unexpected unicode
 --> day11.ipynb:cell 5:1:8
  |
1 | & "C:\Users\Harini\AppData\Local\Programs\Python\Python312\python.exe" -m pip install pandas
  |        ^

invalid-syntax: Simple statements must be separated by newlines or semicolons
 --> day11.ipynb:cell 5:1:75
  |
1 | & "C:\Users\Harini\AppData\Local\Programs\Python\Python312\python.exe" -m pip install pandas
  |                                                                           ^^^

invalid-syntax: Simple statements must be separated by newlines or semicolons
 --> day11.ipynb:cell 5:1:79
  |
1 | & "C:\Users\Harini\AppData\Local\Programs\Python\Python312\python.exe" -m pip install pandas
  |                                                                               ^^^^^^^

invalid-syntax: Simple statements must be separated by 

In [ ]:
# 9. Address review comments and merge.

In [30]:
# 10. Re-run the notebook and show that the second run processes nothing new.
run_pipeline(config)

Skipping already processed file: olist_mini/orders.csv
